# Task 2 (Ablation) — Few-Shot Prompting Experiment
Dataset: TriviaQA (`rc.nocontext`) — Project 10

This is a **separate, standalone experiment**, not the main Task 2 submission notebook.
It reuses the exact same retrieval pipeline as `task2_rag.ipynb`, and changes only the generator prompt: a few (question, answer) exemplars drawn from the TriviaQA `train` split (never used for training -- in-context demonstrations only, no gradient updates) are added to the chat history before the real question, to test whether few-shot formatting guidance improves EM/F1 over the zero-shot prompt used in `task2_rag.ipynb`.

## 0. Overview

Run on Colab (GPU), same as the main Task 2 notebook. Upload `triviaqa_sample_300.jsonl` and `wiki_paragraph_index.jsonl` when prompted. Run locally (no GPU) to reuse an already-downloaded `task2_fewshot_generation_results.jsonl`.

Writes to separate `task2_fewshot_*` result files so it never overwrites the zero-shot Task 2 results -- the two can be compared side by side (see Section 8).

## 1. Setup

In [ ]:
# Importing the libraries we need for this standalone few-shot experiment
import sys
import json
import re
import string
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

# Fixing the seed so this run stays reproducible and comparable to the zero-shot notebook
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Detecting whether we're on Colab (GPU needed for generation) or running locally
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running on Colab:" if IN_COLAB else "Running locally:", IN_COLAB)

In [ ]:
# Installing the Colab-only dependencies (torch/transformers already come
# preinstalled on the GPU runtime, so only the extras are added here)
if IN_COLAB:
    # torch/transformers come preinstalled on Colab GPU runtimes; add the rest.
    get_ipython().system('pip install -q sentence-transformers faiss-cpu accelerate "bitsandbytes>=0.46.1"')
    print("Installed Colab-only dependencies. If this is the FIRST time bitsandbytes was "
          "installed in this session, restart the runtime now (Runtime > Restart session) "
          "and re-run from this cell -- a fresh import is required for its compiled extension "
          "to load correctly.")
else:
    print("Local run: assuming `pip install -r ../requirements.txt` has already been run.")

In [ ]:
# Setting up the config constants used throughout this notebook (same spec as task2_rag.ipynb)
K_LIST = [1, 3, 5, 10]
TOP_K = 5            # spec: dense retriever / generator top-k=5
TOP_K_RETRIEVE = 10  # retrieve enough to also compute recall@10
MAX_PASSAGE_TOKENS = 200   # spec: each passage truncated to 200 tokens before the generator
MAX_NEW_TOKENS = 64        # spec
TEMPERATURE = 0.0          # spec -> greedy decoding
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GEN_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

SAMPLE_FILENAME = "triviaqa_sample_300.jsonl"
INDEX_FILENAME = "wiki_paragraph_index.jsonl"
GENERATION_RESULTS_FILENAME = "task2_fewshot_generation_results.jsonl"

# Loading the fixed question sample and the pooled Wikipedia paragraph index from Task 1,
# either by uploading them on Colab or reading straight from the data/ folder locally
if IN_COLAB:
    from google.colab import files
    print(f"Upload {SAMPLE_FILENAME} and {INDEX_FILENAME} (from this repo's data/ folder):")
    uploaded = files.upload()
    assert SAMPLE_FILENAME in uploaded and INDEX_FILENAME in uploaded, (
        "Both files are required -- re-run this cell and upload them."
    )
    SAMPLE_PATH = Path(SAMPLE_FILENAME)
    INDEX_PATH = Path(INDEX_FILENAME)
    RESULTS_DIR = Path(".")  # Colab session working directory
else:
    DATA_DIR = Path("../../23cs30011_project/data")
    RESULTS_DIR = Path("../../23cs30011_project/results")
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    SAMPLE_PATH = DATA_DIR / SAMPLE_FILENAME
    INDEX_PATH = DATA_DIR / INDEX_FILENAME

GENERATION_RESULTS_PATH = RESULTS_DIR / GENERATION_RESULTS_FILENAME

samples = [json.loads(l) for l in open(SAMPLE_PATH)]
passages = [json.loads(l) for l in open(INDEX_PATH)]
print(f"Loaded {len(samples)} questions (same fixed sample as Task 1).")
print(f"Loaded {len(passages)} paragraphs (same pooled Wikipedia index as Task 1).")

## 2. Dense retrieval: `all-MiniLM-L6-v2` + FAISS flat, cosine similarity

In [ ]:
# Loading the sentence embedding model and FAISS for dense retrieval
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer(EMBED_MODEL_NAME)
print("Embedding dim:", embedder.get_sentence_embedding_dimension())

# Encoding every passage once, up front, so FAISS can build its flat index over them
passage_texts = [p["text"] for p in passages]
passage_embeddings = embedder.encode(
    passage_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

dim = passage_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(passage_embeddings)
print("FAISS flat index built:", faiss_index.ntotal, "vectors, dim", dim)

## 3. Retrieve top-k for every sampled question, evaluate recall@k

In [ ]:
# Encoding every question and retrieving its top-k passages by cosine similarity
question_texts = [s["question"] for s in samples]
question_embeddings = embedder.encode(
    question_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

scores, indices = faiss_index.search(question_embeddings, TOP_K_RETRIEVE)
dense_retrieval_results = [
    list(zip(indices[i].tolist(), scores[i].tolist())) for i in range(len(samples))
]

print("Example dense retrieval for:", samples[0]["question"])
for pid, sc in dense_retrieval_results[0][:3]:
    print(f"  cos_sim={sc:.3f} doc={passages[pid]['doc_title']!r} text={passages[pid]['text'][:120]!r}")

In [ ]:
def normalize_answer(s):
    """SQuAD-style normalization: lowercase, remove punctuation, remove articles, fix whitespace."""
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = " ".join(s.split())
    return s


def recall_at_k(hits, golds, k, passages_ref):
    """Utility function to check whether any gold answer string appears
    (as a normalized substring) in any of the top-k retrieved passages."""
    norm_golds = [normalize_answer(g) for g in golds]
    for pid, _ in hits[:k]:
        norm_text = normalize_answer(passages_ref[pid]["text"])
        if any(g and g in norm_text for g in norm_golds):
            return 1.0
    return 0.0


# Iterating through every question and computing recall@k for each k we track
recall_records = []
for s, hits in zip(samples, dense_retrieval_results):
    rec = {"question_id": s["question_id"]}
    for k in K_LIST:
        rec[f"recall@{k}"] = recall_at_k(hits, s["gold_answers"], k, passages)
    recall_records.append(rec)

recall_df = pd.DataFrame(recall_records)
dense_recall_metrics = {f"recall@{k}": recall_df[f"recall@{k}"].mean() for k in K_LIST}

print("=== Task 2 dense retrieval (all-MiniLM-L6-v2, FAISS flat, cosine) recall@k ===")
for k, v in dense_recall_metrics.items():
    print(f"{k:>10}: {v:.4f}")

## 3b. Pick few-shot exemplars from the TriviaQA `train` split

Per the dataset spec: `train` is a source pool for few-shot exemplars only, never used for training (no weight updates happen here or anywhere else in this project). We draw a small, fixed set of (question, answer) pairs with short canonical answers, to demonstrate the desired concise answer format to the frozen generator via in-context learning.

In [ ]:
# Loading the TriviaQA train split -- used here only as a pool to draw few-shot
# exemplars from, never for fine-tuning (no weight updates anywhere in this notebook)
from datasets import load_dataset

N_FEWSHOT = 3
MAX_FEWSHOT_ANSWER_CHARS = 20  # keep exemplars short, to model the desired concise style

train_ds = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="train")

# Shuffling the train indices with the fixed seed, then walking through them in order
rng_fewshot = random.Random(SEED)
train_indices = list(range(len(train_ds)))
rng_fewshot.shuffle(train_indices)

# Picking the first N_FEWSHOT examples whose answer is short enough to be a good
# concise-format demonstration
fewshot_examples = []
for idx in train_indices:
    row = train_ds[idx]
    ans = row["answer"]["value"]
    if ans and len(ans) <= MAX_FEWSHOT_ANSWER_CHARS:
        fewshot_examples.append({"question": row["question"], "answer": ans})
    if len(fewshot_examples) >= N_FEWSHOT:
        break

print(f"Picked {len(fewshot_examples)} few-shot exemplars from train (seed={SEED}, never used for training):")
for ex in fewshot_examples:
    print(" -", ex)

## 4. Generation: top-5 passages (200 tokens each) -> `Qwen2.5-7B-Instruct` (4-bit), greedy decode

In [ ]:
# Loading and quantizing the generator, and wiring up the few-shot prompt
if IN_COLAB:
    import time
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    torch.manual_seed(SEED)

    # 4-bit quantization config so the 7B model fits comfortably on a single Colab GPU
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )

    print("Loading tokenizer/model:", GEN_MODEL_NAME)
    gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
    gen_model = AutoModelForCausalLM.from_pretrained(
        GEN_MODEL_NAME, quantization_config=quant_config, device_map="auto",
    )
    gen_model.eval()

    def truncate_to_tokens(text, max_tokens):
        """Utility function that truncates text to at most max_tokens generator tokens."""
        ids = gen_tokenizer.encode(text, add_special_tokens=False)
        if len(ids) <= max_tokens:
            return text
        return gen_tokenizer.decode(ids[:max_tokens], skip_special_tokens=True)

    def build_prompt(question, top_passages):
        """Builds the chat prompt: system instruction, few-shot exemplars as prior
        turns, then the real question with its top retrieved passages attached."""

        # Stitching together the top-k retrieved passages into one block of context
        passage_block = "\n\n".join(
            f"Passage {i+1} ({passages[pid]['doc_title']}): "
            f"{truncate_to_tokens(passages[pid]['text'], MAX_PASSAGE_TOKENS)}"
            for i, (pid, _) in enumerate(top_passages[:TOP_K])
        )
        user_msg = (
            "Answer the question using ONLY the information in the passages below. "
            "Respond with just the short answer, no explanation, no full sentence.\n\n"
            f"{passage_block}\n\nQuestion: {question}\nAnswer:"
        )
        # Few-shot: prepend (question, answer) exemplars from `train` as prior chat turns,
        # purely in-context -- no weights are touched here or anywhere in this project.
        messages = [
            {"role": "system", "content": "You are a concise, factual question-answering assistant."},
        ]
        for ex in fewshot_examples:
            messages.append({"role": "user", "content": f"Question: {ex['question']}\nAnswer:"})
            messages.append({"role": "assistant", "content": ex["answer"]})
        messages.append({"role": "user", "content": user_msg})
        return gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def generate_answer(question, top_passages):
        """Builds the prompt for one question and greedily decodes the generator's answer."""
        prompt = build_prompt(question, top_passages)
        inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
        with torch.no_grad():
            out = gen_model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                pad_token_id=gen_tokenizer.eos_token_id,
            )
        new_tokens = out[0][inputs["input_ids"].shape[1]:]
        return gen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Iterating through every sampled question, generating its answer, and logging progress
    generations = []
    t0 = time.time()
    for i, (s, hits) in enumerate(zip(samples, dense_retrieval_results)):
        pred = generate_answer(s["question"], hits)
        generations.append({
            "question_id": s["question_id"], "question": s["question"],
            "gold_answers": s["gold_answers"], "prediction": pred,
        })
        if (i + 1) % 10 == 0:
            elapsed = time.time() - t0
            print(f"{i+1}/{len(samples)} done, {elapsed:.0f}s elapsed, ~{elapsed/(i+1):.1f}s/question")

    # Persisting the generations to their own file, so the zero-shot results are never touched
    with open(GENERATION_RESULTS_PATH, "w") as f:
        for r in generations:
            f.write(json.dumps(r) + "\n")
    print("Saved", GENERATION_RESULTS_PATH)

    print("\nDownload this file and place it in this repo's results/ folder:")
    files.download(str(GENERATION_RESULTS_PATH))

else:
    # Local (no-GPU) run: just reuse a previous Colab run's output for scoring/analysis
    assert GENERATION_RESULTS_PATH.exists(), (
        f"{GENERATION_RESULTS_PATH} not found. Run this notebook on Colab (GPU) first to "
        "generate it, or place a previous run's output there."
    )
    generations = [json.loads(l) for l in open(GENERATION_RESULTS_PATH)]
    print(f"Local run: loaded {len(generations)} pre-generated predictions from "
          f"{GENERATION_RESULTS_PATH} (generation itself requires the Colab GPU run).")

gen_by_qid = {g["question_id"]: g for g in generations}
assert set(gen_by_qid) == {s["question_id"] for s in samples}, "generation set doesn't match the sample set"
print("Question IDs match the fixed sample exactly.")

## 5. Score generations: EM, F1, recall@k

In [ ]:
def compute_em(pred, golds):
    """Utility function to return 1.0 if the prediction exactly matches any
    gold answer (after normalization), else 0.0."""
    norm_pred = normalize_answer(pred)
    return float(any(norm_pred == normalize_answer(g) for g in golds))


def f1_score(pred, gold):
    """Utility function that computes token-level F1 between one prediction
    and one gold answer string."""
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def compute_f1(pred, golds):
    """Utility function that takes the best F1 across all acceptable gold answers."""
    return max(f1_score(pred, g) for g in golds) if golds else 0.0


# Iterating through every question and building one results record per
# question: gold answers, prediction, EM/F1, and recall@k for every k we track
records = []
for s, hits in zip(samples, dense_retrieval_results):
    g = gen_by_qid[s["question_id"]]
    pred = g["prediction"]
    golds = s["gold_answers"]
    rec = {
        "question_id": s["question_id"],
        "question": s["question"],
        "gold_answers": golds,
        "prediction": pred,
        "em": compute_em(pred, golds),
        "f1": compute_f1(pred, golds),
        "top1_doc": passages[hits[0][0]]["doc_title"],
    }
    for k in K_LIST:
        rec[f"recall@{k}"] = recall_at_k(hits, golds, k, passages)
    records.append(rec)

task2_fewshot_results_df = pd.DataFrame(records)
task2_fewshot_results_df.to_csv(RESULTS_DIR / "task2_fewshot_per_question_results.csv", index=False)
task2_fewshot_results_df.head(5)

In [ ]:
# Averaging EM/F1/recall@k over all 300 questions and saving the summary metrics
task2_fewshot_metrics = {
    "n_questions": len(task2_fewshot_results_df),
    "EM": task2_fewshot_results_df["em"].mean(),
    "F1": task2_fewshot_results_df["f1"].mean(),
}
for k in K_LIST:
    task2_fewshot_metrics[f"recall@{k}"] = task2_fewshot_results_df[f"recall@{k}"].mean()

with open(RESULTS_DIR / "task2_fewshot_metrics.json", "w") as f:
    json.dump(task2_fewshot_metrics, f, indent=2)

print("=== Task 2 FEW-SHOT (TriviaQA rc.nocontext) — dense retrieval + LLM generation ===")
for key, val in task2_fewshot_metrics.items():
    print(f"{key:>12}: {val:.4f}" if isinstance(val, float) else f"{key:>12}: {val}")

# side-by-side with Task 1 and the zero-shot Task 2 run, if both are reachable (local run only)
task1_metrics_path = Path("../../results/task1_metrics.json")
task2_zeroshot_metrics_path = Path("../../results/task2_metrics.json")
comparison = {}
if task1_metrics_path.exists():
    with open(task1_metrics_path) as f:
        comparison["Task 1 (BM25 + heuristic)"] = json.load(f)
if task2_zeroshot_metrics_path.exists():
    with open(task2_zeroshot_metrics_path) as f:
        comparison["Task 2 zero-shot (dense + LLM)"] = json.load(f)
comparison["Task 2 few-shot (dense + LLM)"] = task2_fewshot_metrics
if len(comparison) > 1:
    comparison_df = pd.DataFrame(comparison)
    print("\n=== Zero-shot vs few-shot comparison ===")
    print(comparison_df)

if IN_COLAB:
    print("\nDownload these files and place them in this repo's results/ folder:")
    files.download(str(RESULTS_DIR / "task2_fewshot_per_question_results.csv"))
    files.download(str(RESULTS_DIR / "task2_fewshot_metrics.json"))

## 6. Qualitative analysis: correct vs. incorrect examples

In [ ]:
def show_examples(df, n=5, correct=True, seed=SEED):
    """Utility function to print n random correct (EM=1) or incorrect (EM=0)
    examples from the results dataframe, for qualitative inspection."""
    subset = df[df["em"] == (1.0 if correct else 0.0)]
    subset = subset.sample(min(n, len(subset)), random_state=seed)

    # Iterating through the sampled rows and printing question/gold/pred side by side
    for _, row in subset.iterrows():
        print(f"Q: {row['question']}")
        print(f"   Gold: {row['gold_answers']}")
        print(f"   Pred: {row['prediction']!r}")
        print(f"   Top-1 doc: {row['top1_doc']}")
        print(f"   EM={row['em']}  F1={row['f1']:.2f}")
        print("-" * 80)


print(">>> CORRECT (EM=1) examples")
show_examples(task2_fewshot_results_df, n=5, correct=True)

print("\n>>> INCORRECT (EM=0) examples")
show_examples(task2_fewshot_results_df, n=5, correct=False)

## 7. Zero-shot vs. few-shot -- ablation verdict

This notebook's own EM/F1 vs. `task2_rag.ipynb`'s zero-shot EM/F1 (printed in Section 5 above) is the actual comparison that decides which prompting strategy goes into the final submission. Only one of the two (`task2_rag.ipynb` zero-shot, or this few-shot variant) should be kept as the official Task 2 notebook -- whichever scores higher, or ties and is simpler.

## 8. Summary

Run this notebook on Colab to fill in EM/F1/recall@k above, then compare against `results/task2_metrics.json` (zero-shot) to decide whether few-shot replaces the official Task 2 submission notebook.